In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

import config
from etl.data_loader import DataLoader

loader = DataLoader()

print("DATA_ROOT:", config.PathConfig.DATA_ROOT)
print("PROCESSED:", config.PathConfig.PROCESSED)
print("RAW:", config.PathConfig.RAW)

DATA_ROOT: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage
PROCESSED: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage\processed
RAW: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage\raw


In [15]:
processed_files = sorted(Path(config.PathConfig.PROCESSED).glob("*.parquet"))

pd.DataFrame({
    "file": [p.name for p in processed_files],
    "path": [str(p) for p in processed_files],
    "size_mb": [round(p.stat().st_size / 1024 / 1024, 2) for p in processed_files],
})

,file,path,size_mb
0,BNBUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.09
1,BNBUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.20
2,BNBUSDT_1m.parquet,D:\work and study\PostGraduate\HK\project\Trad...,75.01
3,BNBUSDT_4h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.55
4,BTCUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.10
5,BTCUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.65
6,BTCUSDT_1m.parquet,D:\work and study\PostGraduate\HK\project\Trad...,90.24
7,BTCUSDT_4h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.61
8,ETHUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.10
9,ETHUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.52


In [16]:
symbol = "BTC/USDT"
timeframe = "4h"

df = loader.get_crypto_kline_data(
    symbol=symbol,
    timeframe=timeframe,
)

df.head()

,taker_buy_vol,net_taker_vol,close,volume,high,open,low
timestamp,,,,,,,
2021-01-01 00:00:00,21852.599,495.037,29302.11,43210.161,29546.42,28948.19,28706.00
2021-01-01 04:00:00,11772.871,-3136.344,29107.71,26682.086,29422.32,29302.11,28822.00
2021-01-01 08:00:00,14611.792,-339.046,29341.99,29562.630,29454.45,29107.72,28900.00
2021-01-01 12:00:00,23957.859,-1227.234,29210.84,49142.952,29668.86,29342.00,29043.75
2021-01-01 16:00:00,20013.294,-3641.582,29048.47,43668.170,29388.10,29210.85,28627.12


In [17]:
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [df[c].notna().sum() for c in df.columns],
        "missing": [df[c].isna().sum() for c in df.columns],
        "missing_pct": [df[c].isna().mean() for c in df.columns],
        "sample_value": [df[c].dropna().iloc[0] if df[c].dropna().shape[0] else None for c in df.columns],
    })

describe_df(df)

,column,dtype,non_null,missing,missing_pct,sample_value
0,taker_buy_vol,float64,11924,0,0.0,21852.599
1,net_taker_vol,float64,11924,0,0.0,495.037
2,close,float64,11924,0,0.0,29302.110
3,volume,float64,11924,0,0.0,43210.161
4,high,float64,11924,0,0.0,29546.420
5,open,float64,11924,0,0.0,28948.190
6,low,float64,11924,0,0.0,28706.000


In [18]:
pd.DataFrame({
    "symbol": [symbol],
    "timeframe": [timeframe],
    "start": [df.index.min()],
    "end": [df.index.max()],
    "rows": [len(df)],
    "columns": [list(df.columns)],
})

,symbol,timeframe,start,end,rows,columns
0,BTC/USDT,4h,2021-01-01,2026-06-11 04:00:00,11924,"[taker_buy_vol, net_taker_vol, close, volume, ..."


In [24]:
matrix = loader.get_crypto_matrix(
    symbols=["BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT"],
    timeframe="4h",
    columns=["close", "volume", "net_taker_vol"],
)

close = matrix["volume"]
close.tail()

读取加密货币数据 (周期: 4h)...
✅ 成功加载 3 个特征矩阵。


,BTC/USDT,ETH/USDT,SOL/USDT,BNB/USDT
timestamp,,,,
2026-06-09 12:00:00,71792.227,1624864.053,7920044.95,168656.41
2026-06-09 16:00:00,51223.379,1356989.394,6688645.28,100301.68
2026-06-09 20:00:00,17544.777,513326.636,2057497.20,51072.06
2026-06-10 00:00:00,19223.281,565567.199,2573453.63,64341.82
2026-06-10 04:00:00,19399.655,539454.290,2311561.68,81668.74


In [2]:
fund_rate = loader.get_funding_rate_data()
display(fund_rate)

,timestamp,symbol,funding_rate,source,created_at,funding_interval_hours_raw,funding_interval_hours,funding_rate_8h_equiv,funding_rate_chg,funding_rate_z_30_events
0,2021-01-01 00:00:00.002,BNB/USDT,0.000592,binance_usdm,2026-06-11 08:24:56.988878,NaN,<NA>,NaN,NaN,NaN
1,2021-01-01 08:00:00.006,BNB/USDT,0.000568,binance_usdm,2026-06-11 08:24:56.988878,8.000001,8,0.000568,-0.000023,NaN
2,2021-01-01 16:00:00.003,BNB/USDT,0.000873,binance_usdm,2026-06-11 08:24:56.988878,7.999999,8,0.000873,0.000305,NaN
3,2021-01-02 00:00:00.000,BNB/USDT,0.000116,binance_usdm,2026-06-11 08:24:56.988878,7.999999,8,0.000116,-0.000757,NaN
4,2021-01-02 08:00:00.000,BNB/USDT,0.000337,binance_usdm,2026-06-11 08:24:56.988878,8.000000,8,0.000337,0.000221,NaN
...,...,...,...,...,...,...,...,...,...,...
23922,2026-06-10 00:00:00.001,SOL/USDT,0.000018,binance_usdm,2026-06-11 08:24:49.781219,8.000000,8,0.000018,0.000116,0.949003
23923,2026-06-10 08:00:00.011,SOL/USDT,-0.000084,binance_usdm,2026-06-11 08:24:49.781219,8.000003,8,-0.000084,-0.000102,0.078907
23924,2026-06-10 16:00:00.009,SOL/USDT,-0.000018,binance_usdm,2026-06-11 08:24:49.781219,7.999999,8,-0.000018,0.000067,0.705709
23925,2026-06-11 00:00:00.000,SOL/USDT,-0.000082,binance_usdm,2026-06-11 08:24:49.781219,7.999998,8,-0.000082,-0.000065,0.157013
